In [2]:
import cv2
import pytesseract
import numpy as np
import re
import json
import os
from IPython.display import display, JSON  # For structured output in Jupyter

def preprocess_image(image_path):
    """Preprocess the image for better OCR accuracy."""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Error: File not found at {image_path}. Please check the path.")
    
    image = cv2.imread("/Users/hemanthchalla/Desktop/sample.png", cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Error: Unable to read image file {image_path}. It might be corrupted or in an unsupported format.")
    
    # Apply Gaussian Blur to reduce noise
    image = cv2.GaussianBlur(image, (5, 5), 0)
    
    # Convert to binary using OTSU thresholding
    _, thresh = cv2.threshold(image, 150, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    return thresh

def extract_text_from_image(image_path):
    """Extracts text from the preprocessed image using Tesseract OCR."""
    processed_img = preprocess_image(image_path)
    
    # OCR Configuration: --psm 6 treats text as a block
    custom_config = "--psm 6"
    raw_text = pytesseract.image_to_string(processed_img, config=custom_config)
    
    # Clean up text output
    text = re.sub(r'[^A-Za-z0-9.,:\s/-]', '', raw_text)  # Keep alphanumeric and essential symbols
    text = re.sub(r'\s+', ' ', text).strip()  # Normalize spaces
    
    return text

def extract_structured_data(text):
    """Extracts structured data from OCR text using regex."""
    data = {}

    # General Information Extraction
    patient_name_match = re.search(r'Patient Name\s*[:.]?\s*([A-Za-z ]+)', text, re.IGNORECASE)
    dob_match = re.search(r'DOB\s*[:.]?\s*([0-9/]+)', text, re.IGNORECASE)
    date_match = re.search(r'Date\s*[:.]?\s*([0-9/]+)', text, re.IGNORECASE)

    data["patient_name"] = patient_name_match.group(1).strip() if patient_name_match else "Not Provided"
    data["dob"] = dob_match.group(1).strip() if dob_match else "Not Provided"
    data["date"] = date_match.group(1).strip() if date_match else "Not Provided"

    # Injection & Therapy
    injection_match = re.search(r'INJECTION\s*[:.]?\s*(YES|NO)', text, re.IGNORECASE)
    exercise_match = re.search(r'Exercise Therapy\s*[:.]?\s*(YES|NO)', text, re.IGNORECASE)

    data["injection"] = injection_match.group(1).strip() if injection_match else "Not Provided"
    data["exercise_therapy"] = exercise_match.group(1).strip() if exercise_match else "Not Provided"

    # Functional Assessment Extraction
    faq_tasks = {
        "bending": r'Bending or Stooping\s*[:.]?\s*(\d+)',
        "putting_on_shoes": r'Putting on shoes\s*[:.]?\s*(\d+)',
        "sleeping": r'Sleeping\s*[:.]?\s*(\d+)',
        "standing": r'Standing for an hour\s*[:.]?\s*(\d+)',
        "stairs": r'Going up or down a flight of stairs\s*[:.]?\s*(\d+)',
        "walking": r'Walking through a store\s*[:.]?\s*(\d+)',
        "driving": r'Driving for an hour\s*[:.]?\s*(\d+)',
        "preparing_meal": r'Preparing a meal\s*[:.]?\s*(\d+)',
        "yard_work": r'Yard work\s*[:.]?\s*(\d+)',
        "picking_up_items": r'Picking up items off the floor\s*[:.]?\s*(\d+)'
    }

    data["difficulty_ratings"] = {}
    for task, pattern in faq_tasks.items():
        match = re.search(pattern, text, re.IGNORECASE)
        data["difficulty_ratings"][task] = int(match.group(1)) if match else 0  # Default to 0

    # Patient Changes
    patient_changes = {
        "since_last_treatment": r'Patient Changes since last treatment\s*[:.]?\s*(.+)',
        "since_start_of_treatment": r'Patient changes since the start of treatment\s*[:.]?\s*(.+)',
        "last_3_days": r'Describe any functional changes within the last three days\s*[:.]?\s*(.+)'
    }

    data["patient_changes"] = {}
    for key, pattern in patient_changes.items():
        match = re.search(pattern, text, re.IGNORECASE)
        data["patient_changes"][key] = match.group(1).strip() if match else "Not Provided"

    # Pain Symptoms
    pain_symptoms = {
        "pain": r'Pain[:.]?\s*(\d+)',
        "numbness": r'Numbness[:.]?\s*(\d+)',
        "tingling": r'Tingling[:.]?\s*(\d+)',
        "burning": r'Burning[:.]?\s*(\d+)',
        "tightness": r'Tightness[:.]?\s*(\d+)'
    }

    data["pain_symptoms"] = {}
    for key, pattern in pain_symptoms.items():
        match = re.search(pattern, text, re.IGNORECASE)
        data["pain_symptoms"][key] = int(match.group(1)) if match else 0

    # Medical Assistant Data
    ma_data = {
        "blood_pressure": r'Blood Pressure[:.]?\s*([0-9/]+)',
        "hr": r'HR[:.]?\s*(\d+)',
        "weight": r'Weight[:.]?\s*(\d+)',
        "height": r'Height[:.]?\s*([0-9"\']+)',
        "spo2": r'SpO2[:.]?\s*(\d+)',
        "temperature": r'Temperature[:.]?\s*([0-9.]+)',
        "blood_glucose": r'Blood Glucose[:.]?\s*(\d+)',
        "respirations": r'Respirations[:.]?\s*(\d+)'
    }

    data["medical_assistant_data"] = {}
    for key, pattern in ma_data.items():
        match = re.search(pattern, text, re.IGNORECASE)
        data["medical_assistant_data"][key] = match.group(1).strip() if match else "Not Provided"

    return data

def save_json(data, output_file="output.json"):
    """Saves the extracted data to a JSON file and prints it in a structured format."""
    output_json = {"extracted_data": data}

    # Save to file
    with open(output_file, "w") as f:
        json.dump(output_json, f, indent=4)

    # Pretty-print JSON
    print("\nStructured JSON Output:")
    print(json.dumps(output_json, indent=4))

    # Display in Jupyter Notebook
    display(JSON(output_json))

# MAIN EXECUTION
image_path = "/Users/hemanthchalla/Desktop/sample.png"  # Update with actual image path
text = extract_text_from_image(image_path)
extracted_data = extract_structured_data(text)
save_json(extracted_data)



Structured JSON Output:
{
    "extracted_data": {
        "patient_name": "Not Provided",
        "dob": "Not Provided",
        "date": "Not Provided",
        "injection": "YES",
        "exercise_therapy": "YES",
        "difficulty_ratings": {
            "bending": 12345,
            "putting_on_shoes": 1,
            "sleeping": 12345,
            "standing": 0,
            "stairs": 12,
            "walking": 0,
            "driving": 12348,
            "preparing_meal": 12345,
            "yard_work": 123,
            "picking_up_items": 1
        },
        "patient_changes": {
            "since_last_treatment": "Not Provided",
            "since_start_of_treatment": "Rate pein symptoms on a scale of 0-10 10 being the highest: To Be Completed by MA: Blood Freamwe: HI Weight i: Hehe J",
            "last_3_days": "Not Provided"
        },
        "pain_symptoms": {
            "pain": 0,
            "numbness": 0,
            "tingling": 0,
            "burning": 0,
         

<IPython.core.display.JSON object>

In [1]:
import psycopg2

# Database connection details
DB_CONFIG = {
    "dbname": "ocr_db",
    "user": "hemanthchalla",
    "password": "your_password",  # Replace with your actual password
    "host": "localhost",
    "port": "5432"
}

def get_db_connection():
    """Establish and return a database connection."""
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        return conn
    except psycopg2.Error as e:
        print(f"❌ Database connection failed: {e}")
        return None

def verify_tables():
    """Check if required tables exist in the database."""
    required_tables = {"patients", "treatments", "difficulty_ratings", "patient_changes", "pain_symptoms", "medical_assistant_data"}
    conn = get_db_connection()
    
    if not conn:
        return False

    try:
        with conn.cursor() as cursor:
            cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';")
            existing_tables = {row[0] for row in cursor.fetchall()}
            
            missing_tables = required_tables - existing_tables
            if missing_tables:
                print(f"⚠️ Warning: The following tables are missing: {missing_tables}")
                return False
            
            print("✅ All required tables exist!")
            return True

    except psycopg2.Error as e:
        print(f"❌ Error verifying tables: {e}")
        return False
    finally:
        conn.close()

def insert_test_data():
    """Insert a test record into the patients table and verify the insertion."""
    conn = get_db_connection()
    
    if not conn:
        return

    try:
        with conn.cursor() as cursor:
            # Insert test data
            cursor.execute("""
                INSERT INTO patients (name, dob, assessment_date) 
                VALUES (%s, %s, %s) RETURNING patient_id;
            """, ('Test Patient', '1990-01-01', '2025-02-12'))
            
            patient_id = cursor.fetchone()[0]
            conn.commit()
            print(f"✅ Test patient inserted successfully with ID: {patient_id}")

            # Verify the data was inserted
            cursor.execute("SELECT name, dob, assessment_date FROM patients WHERE patient_id = %s;", (patient_id,))
            result = cursor.fetchone()

            if result:
                print(f"✅ Verification Passed! Data retrieved: {result}")
            else:
                print("❌ Verification Failed! Data not found.")

    except psycopg2.Error as e:
        print(f"❌ Database error: {e}")
    finally:
        conn.close()

# Run verification checks
if __name__ == "__main__":
    print("\n🔍 Running Database Verification Checks...\n")
    
    if verify_tables():
        insert_test_data()
    
    print("\n✅ Database verification completed!\n")



🔍 Running Database Verification Checks...

⚠️ Warning: The following tables are missing: {'patient_changes', 'difficulty_ratings', 'patients', 'pain_symptoms', 'treatments', 'medical_assistant_data'}

✅ Database verification completed!

